## Deep Q-Network (DQN) Agent: Atari Centipede
Author: Raymond Xie

This code trains an AI model to play the Atari game, Centipede, using a Deep Q-Network (DQN) model.<br>
It uses Stable-Baselines3 DQN with a CNN Policy (used for all Atari AI agents).

### Imports
Install Gymnasium Atari, Stable-Baselines3 and MoviePy

In [ ]:
# Install libraries (Gymnasium Atari + Stable-Baselines3) for Centipede game and RL algorithm
# If you have these libraries installed already, you do not need to run this cell
%pip install -q "gymnasium[atari, accept-rom-license]" "stable-baselines3[extra]"

In [ ]:
# Install MoviePy for the recording of the agent training process
# Run this cell only if you do not already have MoviePy installed on your device
%pip install -q "moviepy"

### Create Game Environment
Importing the game environment and creating 4 parallel environments for faster training.

In [ ]:
# Import statements
import gymnasium as gym
import ale_py
import numpy as np

# Importing the model
from stable_baselines3 import DQN

print("Gymnasium version:", gym.__version__)

# Using Arcade Learning Environment for Centipede
env_id = "ALE/Centipede-v5"

env = gym.make(env_id, render_mode=None)
obs, info = env.reset()
print("Obs shape:", obs.shape)
print("Action space:", env.action_space)
env.close()


In [ ]:
# Creating Centipede environment
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

env_id = "ALE/Centipede-v5"

# Variable for how many simultaneous environments we create to train the AI faster
n_envs = 4

# Creates n_envs environments and runs the environments
vec_env = make_atari_env(env_id, n_envs=n_envs, seed=0)
vec_env = VecFrameStack(vec_env, n_stack=4)
vec_env


### Tensorboard (Optional)
Creates a Tensorboard to show the average reward of each timestep during training.<br>
This cell can be deleted if you do not wish to view the training process or if it is not loading correctly.

In [ ]:
%reload_ext tensorboard
%load_ext tensorboard
%tensorboard --logdir ./centipede_logs --reload_interval 30

### Define DQN Model Parameters
We are using CNN Policy which is commonly used for Atari games.<br>

Important Parameters:
- <code>total_timesteps</code>: Specifies how many timesteps you wish to train the agent for.
- <code>learning_rate</code>: The learning rate the agent will learn at.
- <code>buffer_size</code>: Used in DQN, specifies the max number of timesteps stored in memory to learn from.
- <code>learning_starts</code>: Used in DQN, specifies the number of timesteps to go through before training begins.
- <code>batch_size</code>: Used in DQN, specifies the number of training examples per update to the policy.

In [ ]:
from stable_baselines3.common.callbacks import CheckpointCallback

# The total number of timesteps the agent is allowed to interact with the environment for
total_timesteps = 100_000
# The learning rate the agent will learn at, 2.5e-4 is more stable for DQN
learning_rate = 2.5e-4

# Defining the model
# CNN policy is the policy the AI should use when learning (CNN is used for all AIs learning Atari games)
model = DQN(
    "CnnPolicy",
    vec_env,
    tensorboard_log="./centipede_logs",
    verbose=1,
    learning_rate=learning_rate,
    # Will store 100,000 timesteps at a time
    buffer_size=100000,
    # Will start learning when 50,000 timesteps have been stored
    learning_starts=50000,
    # Will use 32 training examples per update
    batch_size=32,
    train_freq=4,
    target_update_interval=10000,
    gamma=0.99,
)

# Saves the model every 100,000 timesteps 
checkpoint_callback = CheckpointCallback(
    save_freq=100_000,
    save_path="./centipede_checkpoints/",
    name_prefix="dqn_centipede",
    save_replay_buffer=False,
    save_vecnormalize=False,
)

# Making the AI interact with the environment to learn
model.learn(
    total_timesteps=total_timesteps,
    callback=checkpoint_callback,
)

# Saving the model and informing user that training has been finished
model.save("dqn_centipede_final")
print("Training done")


### Evaluate Trained Agent
We run several episodes to compute the mean reward the trained agent will obtain.

In [ ]:
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

# Creating a new environment for the trained AI
eval_env = make_atari_env(env_id, n_envs=1, seed=42)
eval_env = VecFrameStack(eval_env, n_stack=4)

# Letting the AI run 10 times, then outputting the average rewards it had during those 10 runs
mean_reward, std_reward = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=10,
    deterministic=True,
    render=False,
)

# Prints out the average reward over 10 episodes with standard deviation
print(f"Mean reward over 10 episodes: {mean_reward:.2f} ± {std_reward:.2f}")

# Closing the environment
eval_env.close()


### Saving Training Video (Optional)
Saves a video of the trained agent playing Centipede.<br>
Completely optional, keep only if you wish to view the results of the training.

In [ ]:
import os
from stable_baselines3.common.vec_env import VecVideoRecorder

# Recording a video of the AI
video_folder = "centipede_videos"
video_length = 1_000

os.makedirs(video_folder, exist_ok=True)

# Create a new environment for the video
video_env = make_atari_env(env_id, n_envs=1, seed=123)
video_env = VecFrameStack(video_env, n_stack=4)

# Details on where to store the video and how long it is
video_env = VecVideoRecorder(
    video_env,
    video_folder,
    record_video_trigger=lambda step: step == 0,
    video_length=video_length,
    name_prefix="dqn-centipede",
)

obs = video_env.reset()
for step in range(video_length + 1):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = video_env.step(action)

# Closing the environment made for the video
video_env.close()

# Letting the user know where the video was saved
print("Video recorded to:", video_folder)
